# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nandhanamj/flyrank_ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

###1. Method choice

I will use Logistic Regression because this lane is a prioritization problem where the goal is to score content items for refresh review.

The model is simple and interpretable, and its predicted probability can be used to rank content items. This gives a learned scoring baseline that can be compared with my Week-4 hand-written rule.

I will use only features available at the March 31, 2026 decision point and will evaluate the model using the same Precision@50 metric as my Week-4 baseline.

In [1]:
# Setup for Week-5 modeling

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score

print("Libraries loaded.")

Libraries loaded.


In [3]:
# DuckDB setup

import duckdb

con = duckdb.connect()

print("DuckDB connection ready.")

DuckDB connection ready.


In [6]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


###2. Split design

I will use a client-grouped train/test split with 80% of the rows for training and 20% for testing.

Grouping by client prevents content from the same client appearing in both sets. This makes the evaluation more honest for the question of whether the model can generalize to clients it did not see during training.

The test set will be held out until model evaluation, and both the Logistic Regression model and my Week-4 baseline will be evaluated on this same test set using Precision@50.

In [8]:
# Build the modeling dataset
# March = features available at the decision point
# April = future window used only to create the label

DIM_CONTENT = """
read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
"""

PERF_MARCH = """
read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
"""

PERF_APRIL = """
read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet')
"""

model_data = con.execute(f"""
WITH content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_updated_date,
        search_volume,
        word_count,
        backlinks
    FROM {DIM_CONTENT}
    WHERE content_updated_date IS NOT NULL
      AND is_deleted IS NOT TRUE
),

march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        AVG(NULLIF(gsc_avg_position, 0)) AS march_avg_position,
        SUM(ga4_pageviews) AS march_pageviews,
        SUM(ga4_engaged_sessions) AS march_engaged_sessions,
        BOOL_OR(gsc_data_available IS TRUE) AS march_gsc_available
    FROM {PERF_MARCH}
    GROUP BY 1, 2
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS april_clicks,
        BOOL_OR(gsc_data_available IS TRUE) AS april_gsc_available
    FROM {PERF_APRIL}
    GROUP BY 1, 2
)

SELECT
    c.client_hash_id,
    c.content_hash_id,

    DATE_DIFF(
        'day',
        c.content_updated_date,
        DATE '2026-03-31'
    ) AS days_stale,

    m.march_impressions,
    m.march_clicks,
    m.march_avg_position,
    m.march_pageviews,
    m.march_engaged_sessions,

    c.search_volume,
    c.word_count,
    c.backlinks,

    a.april_clicks,

    CASE
        WHEN a.april_clicks < m.march_clicks THEN 1
        ELSE 0
    END AS is_declining_future

FROM content c
JOIN march m
    USING (client_hash_id, content_hash_id)
JOIN april a
    USING (client_hash_id, content_hash_id)

WHERE m.march_gsc_available
  AND a.april_gsc_available
""").df()

print(f"Modeling rows: {len(model_data):,}")
print(f"Clients: {model_data['client_hash_id'].nunique():,}")
print("\nFuture-label distribution:")
print(model_data["is_declining_future"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 158,466
Clients: 45

Future-label distribution:
is_declining_future
0    114171
1     44295
Name: count, dtype: int64


In [9]:
# Create a client-grouped train/test split

from sklearn.model_selection import GroupShuffleSplit

FEATURES = [
    "days_stale",
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_pageviews",
    "march_engaged_sessions",
    "search_volume",
    "word_count",
    "backlinks",
]

TARGET = "is_declining_future"
GROUP = "client_hash_id"

X = model_data[FEATURES]
y = model_data[TARGET]
groups = model_data[GROUP]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Training clients: {len(train_clients)}")
print(f"Test clients: {len(test_clients)}")
print(f"Client overlap: {len(train_clients & test_clients)}")

Training rows: 125,592
Test rows: 32,874
Training clients: 36
Test clients: 9
Client overlap: 0


###3. Train + compare vs my baseline

I will train Logistic Regression using only March decision-point features. Missing numeric values will be median-imputed, and features will be standardized before fitting.

The model will produce a probability of future decline for each content item. I will use that probability to rank the test-set items.

For a fair comparison, my Week-4 baseline will be rebuilt using the same test rows and the same March decision-point signals. Both approaches will then be compared using Precision@50.

In [10]:
# Train Logistic Regression

from sklearn.impute import SimpleImputer

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

model_scores = model.predict_proba(X_test)[:, 1]

print("Logistic Regression trained.")
print(f"Test predictions: {len(model_scores):,}")
print(f"Mean predicted decline probability: {model_scores.mean():.3f}")

Logistic Regression trained.
Test predictions: 32,874
Mean predicted decline probability: 0.252


In [11]:
# Recreate the Week-4 baseline on the held-out test set

test_results = model_data.iloc[test_idx].copy()

test_results["model_score"] = model_scores

# Week-4 staleness score
test_results["staleness_score"] = np.select(
    [
        test_results["days_stale"] >= 365,
        test_results["days_stale"] >= 180,
        test_results["days_stale"] >= 90,
    ],
    [3, 2, 1],
    default=0
)

# Week-4 visibility score
test_results["visibility_score"] = np.select(
    [
        test_results["march_impressions"] < 100,
        test_results["march_impressions"] < 1000,
        test_results["march_impressions"] < 10000,
    ],
    [3, 2, 1],
    default=0
)

test_results["baseline_score"] = (
    test_results["staleness_score"]
    + test_results["visibility_score"]
)

# Rank both approaches on exactly the same test rows
test_results["model_rank"] = (
    test_results["model_score"]
    .rank(method="first", ascending=False)
)

test_results["baseline_rank"] = (
    test_results["baseline_score"]
    .rank(method="first", ascending=False)
)

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-np.asarray(scores))
    top_k = order[:k]
    return np.mean(np.asarray(y_true)[top_k])

model_p50 = precision_at_k(
    test_results["is_declining_future"],
    test_results["model_score"],
    k=50
)

baseline_p50 = precision_at_k(
    test_results["is_declining_future"],
    test_results["baseline_score"],
    k=50
)

comparison = pd.DataFrame({
    "method": ["Week-4 baseline", "Logistic Regression"],
    "precision_at_50": [baseline_p50, model_p50]
})

display(comparison)

print(f"Baseline Precision@50: {baseline_p50:.3f}")
print(f"Model Precision@50:    {model_p50:.3f}")

,method,precision_at_50
0,Week-4 baseline,0.08
1,Logistic Regression,0.74


Baseline Precision@50: 0.080
Model Precision@50:    0.740


### Model vs baseline

On the held-out test set, the Week-4 baseline achieved Precision@50 of 0.080, while Logistic Regression achieved Precision@50 of 0.740.

Both methods were evaluated on the same test rows using the same future-decline label and Precision@50 metric.

The model therefore identified a substantially higher proportion of future-declining content among its top 50 ranked items in this measured test split. This is an observed test-set result, not evidence that the model will always perform this well on future data.

###4. Errors and interpretation

I will inspect the model's top 50 test-set predictions and separate correct positive predictions from false positives.

I will also inspect the model coefficients to understand which features the Logistic Regression model leans on. The goal is to understand the types of mistakes and the main observed signals behind the ranking, rather than judging the model only by its headline metric.

In [12]:
# Inspect top-50 model predictions and model coefficients

top50_model = (
    test_results
    .sort_values(
        ["model_score", "content_hash_id"],
        ascending=[False, True]
    )
    .head(50)
    .copy()
)

top50_model["prediction"] = (
    top50_model["is_declining_future"]
    .map({1: "CORRECT_POSITIVE", 0: "FALSE_POSITIVE"})
)

print("Top-50 model predictions:")
display(
    top50_model[
        [
            "content_hash_id",
            "model_score",
            "is_declining_future",
            "prediction",
            "days_stale",
            "march_impressions",
            "march_clicks",
            "march_avg_position",
        ]
    ].head(10)
)

print("\nTop-50 error counts:")
print(top50_model["prediction"].value_counts())

# Logistic Regression coefficients
coefficients = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": model.named_steps["logistic"].coef_[0]
})

coefficients["absolute_coefficient"] = coefficients["coefficient"].abs()

coefficients = coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
)

print("\nModel coefficients:")
display(coefficients)

Top-50 model predictions:


,content_hash_id,model_score,is_declining_future,prediction,days_stale,march_impressions,march_clicks,march_avg_position
85969,content_eadb33b5df496f4a,1.000000,0,FALSE_POSITIVE,-73,617124.0,5668.0,2.383011
87080,content_ec2e0346994fb5a5,1.000000,1,CORRECT_POSITIVE,-73,245276.0,1480.0,2.854514
73258,content_0e03de7680314cd5,1.000000,0,FALSE_POSITIVE,-73,221310.0,720.0,2.675217
73253,content_4ffe18112a5642e3,1.000000,0,FALSE_POSITIVE,-73,186983.0,586.0,2.331060
73224,content_8d7d99f109e19aa2,1.000000,0,FALSE_POSITIVE,-73,203497.0,289.0,2.563756
73252,content_545bb6cc7081ded3,1.000000,0,FALSE_POSITIVE,-73,122905.0,287.0,2.615390
86231,content_18f0847d6628f8c6,1.000000,1,CORRECT_POSITIVE,-73,48911.0,703.0,3.825143
105836,content_f86f77b3ebdc05ee,0.999999,1,CORRECT_POSITIVE,-85,105420.0,548.0,3.942110
94000,content_77276ad7a26f4905,0.999999,0,FALSE_POSITIVE,-83,116707.0,200.0,3.917468
105826,content_963de14b1f58978f,0.999997,1,CORRECT_POSITIVE,-83,97312.0,482.0,3.753124



Top-50 error counts:
prediction
CORRECT_POSITIVE    37
FALSE_POSITIVE      13
Name: count, dtype: int64

Model coefficients:


,feature,coefficient,absolute_coefficient
3,march_avg_position,-0.679479,0.679479
1,march_impressions,0.656747,0.656747
7,word_count,0.136109,0.136109
0,days_stale,-0.105106,0.105106
5,march_engaged_sessions,0.080141,0.080141
2,march_clicks,0.061583,0.061583
6,search_volume,-0.042372,0.042372
4,march_pageviews,0.027370,0.027370
8,backlinks,0.006759,0.006759


### Error interpretation

Among the model's top 50 test-set predictions, 37 were correct positive predictions and 13 were false positives, giving the measured Precision@50 of 0.740.

The model's largest coefficient magnitude was for `march_avg_position`, followed by `march_impressions`. This indicates that these features had the strongest measured relationship with the model's predicted probability within this fitted Logistic Regression model.

The false positives show that a high model score does not guarantee future decline. For example, some highly ranked false positives had substantial March impressions and clicks, so the model can identify pages with strong observed search activity that nevertheless did not decline in the following month.

I also observed negative `days_stale` values in some top-ranked rows. These indicate content update dates after the March 31 decision point and should be treated as a data-quality issue rather than interpreted as genuine staleness. This is a limitation of the current modeling dataset that should be checked before using the feature in production.

Overall, the test result is a measured comparison on the held-out clients. It supports further investigation of the learned ranking, but it does not establish that the same Precision@50 will hold on future clients or future time periods.

In [13]:
# Final leakage and evaluation checks

future_terms = [
    "april",
    "future",
    "label",
    "declining_future"
]

feature_leakage = [
    feature for feature in FEATURES
    if any(term in feature.lower() for term in future_terms)
]

print("Features used by the model:")
print(FEATURES)

print("\nFuture/label-derived feature names detected:")
print(feature_leakage)

print("\nTrain/test client overlap:")
print(len(train_clients & test_clients))

print("\nFinal evaluation:")
print(f"Baseline Precision@50: {baseline_p50:.3f}")
print(f"Model Precision@50:    {model_p50:.3f}")

Features used by the model:
['days_stale', 'march_impressions', 'march_clicks', 'march_avg_position', 'march_pageviews', 'march_engaged_sessions', 'search_volume', 'word_count', 'backlinks']

Future/label-derived feature names detected:
[]

Train/test client overlap:
0

Final evaluation:
Baseline Precision@50: 0.080
Model Precision@50:    0.740


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, private URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] The model and Week-4 baseline were evaluated on the same held-out test set and Precision@50 metric
- [x] Client-grouped split has zero client overlap between train and test
- [x] Future April data is used only to create the evaluation label, not as a model feature
- [x] Committed to my repo under `work/notebooks/` — then submit my repo URL on the card